In [2]:
import cv2
import numpy as np
from pathlib import Path



dir_path = Path('./images')
imgs_path = [ str(f) for f in dir_path.iterdir() if f.is_file() ]


print(imgs_path)

['images/poor_details.png', 'images/Shadows.png', 'images/low_contrast.png']


In [3]:
imgs = [cv2.imread(img) for img in imgs_path ]
original_imgs = [cv2.resize(img, dsize=None, fx=0.1, fy=0.1) for img in imgs]

for p, img in zip(imgs_path, original_imgs):
    cv2.imshow(p + "_resized", img)


cv2.waitKey(0)
cv2.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/emagira/uni/CV/Assignments_Computer_Vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/emagira/uni/CV/Assignments_Computer_Vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/emagira/uni/CV/Assignments_Computer_Vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/emagira/uni/CV/Assignments_Computer_Vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fon

In [4]:
blurred_imgs = [cv2.GaussianBlur(img, (11,11), 0) for img in original_imgs]
for p, img in zip(imgs_path, blurred_imgs):
    cv2.imshow(p + "blurred", img)


cv2.waitKey(0)
cv2.destroyAllWindows()

In [5]:
a = 0.5

# we set beta to push back on the darkening of the image
b = 128 * (1 - a)

reduces_contrast_imgs = [cv2.convertScaleAbs(src=img, dst=None, alpha=a, beta=b) for img in blurred_imgs]

for p, img in zip(imgs_path, reduces_contrast_imgs):
    cv2.imshow(p + "reduced_contrast", img)


degraded_imgs = reduces_contrast_imgs

cv2.waitKey(0)
cv2.destroyAllWindows()


In [6]:
#BASELINE 

out_dir = Path('./results/degraded_images')
out_dir.mkdir(parents=False, exist_ok=True)
for p, img in zip(imgs_path, degraded_imgs):
    out_path = out_dir / (Path(p).stem + "_degraded.png")   
    ok = cv2.imwrite(str(out_path), img)
    print(out_path, ok)


results/degraded_images/poor_details_degraded.png True
results/degraded_images/Shadows_degraded.png True
results/degraded_images/low_contrast_degraded.png True


In [7]:
#LINEAR CONTRAST STRETCHING


def stretch_channel(c):
  f_min = np.min(c)
  f_max = np.max(c)

  if f_max == f_min:
    return c.copy()

  stretched = (c.astype(np.float32) - f_min) / (f_max - f_min) * 255.0
  return np.clip(stretched, 0, 255).astype(np.uint8)


def linear_contrast_stretching_per_channel(image):
  
  out = np.empty_like(image)
  for c in range(image.shape[2]):
    out[:,:,c] = stretch_channel(image[:,:,c])
  return out


def linear_contrast_stretching_luminance(image):
  lab_img = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
  lab_img[:,:,0] = stretch_channel(lab_img[:,:,0])  
  return cv2.cvtColor(lab_img, cv2.COLOR_LAB2BGR)


stretched_imgs = [linear_contrast_stretching_per_channel(img) for img in degraded_imgs]


out_dir = Path('./results/linear_contrast_stretching')
out_dir.mkdir(parents=False, exist_ok=True)
for p, img in zip(imgs_path, stretched_imgs):
    out_path = out_dir / (Path(p).stem + "_streched_channel.png")
    ok = cv2.imwrite(str(out_path), img)
    print(out_path, ok)


stretched_lum_imgs = [linear_contrast_stretching_luminance(img) for img in degraded_imgs]

out_dir = Path('./results/linear_contrast_stretching')
for p, img in zip(imgs_path, stretched_lum_imgs):
    out_path = out_dir / (Path(p).stem + "_streched_luminance.png")
    ok = cv2.imwrite(str(out_path), img)
    print(out_path, ok)

# linear stretching fails on poor_details.png: its range is already almost full (bright lights), so it only undoes the degradation; the histogram stays crowded in the dark end


results/linear_contrast_stretching/poor_details_streched_channel.png True
results/linear_contrast_stretching/Shadows_streched_channel.png True
results/linear_contrast_stretching/low_contrast_streched_channel.png True
results/linear_contrast_stretching/poor_details_streched_luminance.png True
results/linear_contrast_stretching/Shadows_streched_luminance.png True
results/linear_contrast_stretching/low_contrast_streched_luminance.png True


In [ ]:
#Histogram equalization


def histogram_equalization_hsv(image):
  hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
  hsv[:,:,2] = cv2.equalizeHist(hsv[:,:,2])   
  return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def histogram_equalization_lab(image):
  lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
  lab[:,:,0] = cv2.equalizeHist(lab[:,:,0]) 
  return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


hist_eq_hsv_imgs = [histogram_equalization_hsv(img) for img in degraded_imgs]
hist_eq_lab_imgs = [histogram_equalization_lab(img) for img in degraded_imgs]


out_dir = Path('./results/histogram_equalization')
out_dir.mkdir(parents=False, exist_ok=True)
for p, img in zip(imgs_path, hist_eq_hsv_imgs):
    out_path = out_dir / (Path(p).stem + "_hist_equal_hsv.png")
    ok = cv2.imwrite(str(out_path), img)
    print(out_path, ok)

out_dir = Path('./results/histogram_equalization')
out_dir.mkdir(parents=False, exist_ok=True)
for p, img in zip(imgs_path, hist_eq_lab_imgs):
    out_path = out_dir / (Path(p).stem + "_hist_equal_lab.png")
    ok = cv2.imwrite(str(out_path), img)
    print(out_path, ok)


#Hist equal, seems to amplify Noise

results/histogram_equalization/poor_details_hist_equal_hsv.png True
results/histogram_equalization/Shadows_hist_equal_hsv.png True
results/histogram_equalization/low_contrast_hist_equal_hsv.png True
results/histogram_equalization/poor_details_hist_equal_lab.png True
results/histogram_equalization/Shadows_hist_equal_lab.png True
results/histogram_equalization/low_contrast_hist_equal_lab.png True


In [ ]:
#Bilateral filter + histogram equalization (LAB)
#we try a denoise before hist equalization

def bilateral_then_equalize(image, d=9, sigma_color=30, sigma_space=5):
  denoised = cv2.bilateralFilter(src=image, d=d, sigmaColor=sigma_color, sigmaSpace=sigma_space)
  return histogram_equalization_lab(denoised)

combined_imgs = [bilateral_then_equalize(img) for img in degraded_imgs]

out_dir = Path('./results/bilateral_filter')
out_dir.mkdir(parents=False, exist_ok=True)
for p, img in zip(imgs_path, combined_imgs):
    out_path = out_dir / (Path(p).stem + "_denoise_and_equalize.png")
    ok = cv2.imwrite(str(out_path), img)
    print(out_path, ok)

results/bilateral_filter/poor_details_denoise_and_equalize.png True


results/bilateral_filter/Shadows_denoise_and_equalize.png True
results/bilateral_filter/low_contrast_denoise_and_equalize.png True
